# Práctica 1

En este ejercicio se va a crear un sistema de aprendizaje supervisado sobre el dataset [ISOLET](https://archive.ics.uci.edu/dataset/54/isolet). Ten en cuenta que este es un problema de clasificación multiclase. Las etiquetas son números enteros que indican la clase. Tu modelo debe predecir un vector que represente una distribución de probabilidad a lo largo de las distintas clases. La función de pérdida que debes utilizar es [nn.CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). Fíjate en que esta función espera que la salida de tu red no esté normalizada con una función softmax (lee la documentación y haz pruebas para aprender a utilizarla correctamente).

## Carga del dataset

Esta celda se encargará de bajar el dataset a vuestro ordenador. Tened en cuenta que puede haber fallos, ya que el repositorio de UCI desde donde se descarga tiene muchas peticiones simultáneas.

In [ ]:
import os
import numpy as np
from scipy.io import loadmat
import torch
import random
from sklearn.model_selection import KFold

# Fijamos la semilla para poder reproducir los resultados
seed = 1234
os.environ['PYTHONHASHSEED'] = str(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

def download_dataset(directory):
    if not os.path.isdir(directory):
        os.makedirs(directory)

    filename = 'Isolet.mat'
    url = 'https://jundongl.github.io/scikit-feature/files/datasets/Isolet.mat'
    filepath = os.path.join(directory, filename)

    if not os.path.isfile(filepath):
        torch.hub.download_url_to_file(url, filepath)


def load_dataset(directory=None):
    directory = directory or './isolet/'
    download_dataset(directory=directory)
    dataset = load_data(directory + 'Isolet.mat')
    return dataset


def load_data(source):
    dataset = loadmat(source)
    return dataset['X'].astype(np.float32), dataset['Y'].astype(int)


In [ ]:
from sklearn.model_selection import train_test_split
import torch.utils.data as data

# Cargamos el dataset
X, y = load_dataset()
y = y - 1
n_classes = np.unique(y).size

# Usa estas particiones exactamente
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

## Preprocesado

Preprocesa los datos como necesites, con el objetivo de facilitar el entrenamiento de la red. Recuerda que, si vas a utilizar valores estadísticos de los datos, éstos sólo pueden ser obtenidos de los datos de la partición de train. Es decir, **no se puede aplicar la función fit (o fit_transform) sobre los datos de validación o test**.

In [ ]:
# TODO - Haz el preprocesado que necesites

## Trabajo a realizar

```
 NOTA: Para todos los entrenamientos se utilizarán los siguientes parámetros:
 - Adam como optimizador
 - Tasa de aprendizaje de 0.001
 - Tamaño de batch de 100
 - Duración de 100 iteraciones (epochs).
```

### 1. Entrenamiento simple
Entrena un modelo de red neuronal totalmente conectada. El número de capas ocultas será de 2, con 50 neuronas por capa, una regularización l2 de 0.01, y ReLU como función de activación para las capas ocultas. Utiliza [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) para el particionamiento, probando con distintos valores de k (3, 5 y 10).

Debes ser capaz de responder a las siguientes preguntas:
1. ¿Cuál es la precisión del modelo?
1. ¿Qué conclusiones sacas respecto a la variación de los resultados al cambiar k?

In [ ]:
# TODO - Entrenamiento simple

### 2. Búsqueda de mejores hiperparámetros
Para un k-fold con k=3, queremos buscar un modelo que se adecúe de la mejor manera posible al problema. Para ello, probaremos distintas configuraciones. En concreto, se entrenarán modelos con todas las variaciones posibles de los parámetros siguientes:
1. 1, 5 o 10 capas ocultas 
1. 10, 50 o 100 número de neuronas de las capas ocultas (todas las capas usarán el mismo valor)
1. 4 valores distintos de regularización por norma l2 (0.0001, 0.001, 0.01 o 0.1).
1. 2 funciones de activación distintas para la capa oculta (ReLU o Tanh).

Asegúrate de que:
 - Se han realizado todas las permutaciones posibles
 - No hay repeticiones innecesarias de código
 - No se ha utilizado el conjunto de validación para entrenar el modelo (ni como criterio de parada)
 - Al final de los entrenamientos, se muestra la precisión (accuracy) del mejor y del peor modelo, así como los parámetros que se han utilizado para crear dichos modelos. También se mostrará la precisión del modelo obtenida con los datos usados en entrenamiento.
 - Para los mejores y peores modelos, investiga si están sobreentrenando o si les falta capacidad.
 - Identificas qué hiperparámetros afectan más y cuáles menos al entrenamiento.

In [2]:
activations = ['relu', 'tanh']
regularizations = [1e-4, 1e-3, 1e-2, 1e-1]
hidden_layer_elements_list = [10, 50, 100]
n_hidden_layers_list = [1, 5, 10]

# Registra los resultados en estos diccionarios respetando el formato dado
# Se incluye un ejemplo con datos inventados para mostrar el formato
resultados_train = {}
resultados_validacion = {}

# Ejemplo de cómo almacenar los resultados en el formato necesario
# TODO - Eliminar cuando se haya probado
for i in activations:
    resultados_validacion[i] = {}
    for j in regularizations:
        resultados_validacion[i][j] = {}
        for k in hidden_layer_elements_list:
            resultados_validacion[i][j][k] = {}
            for l in n_hidden_layers_list:
                resultados_validacion[i][j][k][l] = j * k * l
# Fin del código de ejemplo

#TODO - Búsqueda de hiperparámetros

### 3. Visualiza los resultados

Usa el código de la siguiente celda para visualizar la precisión mediante el uso de mapas de calor. Usa las siguientes combinaciones:
1. Número de capas ocultas vs Número de neuronas por capa
1. Número de capas ocultas vs Regularización
1. Número de neuronas por capa vs Regularización

```Vigila el no repetir código innecesariamente. No se deben re-entrenar los modelos, ya que las estadísticas deben haber sido guardadas en el formato apropiado en el punto anterior.```
 

Analiza los resultados. ¿Qué problemáticas entiendes que se están produciendo?

In [ ]:
# Se facilita el código de generación del mapa de calor a partir de los resultados registrados

import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np

def plot_heatmap_with_values(results, grids, xaxis, yaxis, naxis, activation, fmt="{:.3f}", log_y=False):
    X = np.array(grids[xaxis])
    Y = np.array(grids[yaxis])


    fig, ax_t = plt.subplots(1,len(grids[naxis]), figsize=(16,4))
    for n, nv in enumerate(grids[naxis]):
        ax = ax_t[n]
        # Build Z grid
        Z = []
        comb = {}
        comb['act'] = activation
        comb[naxis] = nv
        for yval in grids[yaxis]:
            comb[yaxis] = yval
            Z_row = []
            for xval in grids[xaxis]:
                comb[xaxis] = xval
                Z_row.append(results[comb['act']][comb['reg']][comb['hid_n']][comb['n_hid']])
            Z.append(Z_row)
        Z = np.array(Z)

        # Compute cell centers along X and Y
        x_edges = np.linspace(X.min(), X.max(), Z.shape[1]+1)
        y_edges = np.linspace(Y.min(), Y.max(), Z.shape[0]+1)
        x_centers = (x_edges[:-1] + x_edges[1:]) / 2
        y_centers = (y_edges[:-1] + y_edges[1:]) / 2

        if log_y:
            ax.set_yscale('log')
            # Compute edges in log space
            y_edges = np.logspace(np.log10(Y.min()), np.log10(Y.max()), Z.shape[0]+1)
            # Compute centers in log space
            y_centers = (y_edges[:-1] * y_edges[1:])**0.5   # geometric mean
        
        pc = ax.pcolormesh(x_edges, y_edges, Z, cmap=cm.coolwarm, shading='auto')

        # Add numeric labels at cell centers
        for i in range(Z.shape[0]):
            for j in range(Z.shape[1]):
                ax.text(x_centers[j], y_centers[i], fmt.format(Z[i, j]),
                        ha="center", va="center", color="black", fontsize=11)

        ax.set_xlabel(xaxis)
        ax.set_ylabel(yaxis)
        ax.set_title(f'{naxis}={nv}')
        fig.colorbar(pc, ax=ax, shrink=0.7)

    plt.show()

grids = {'act':activations, 'reg':regularizations, 'hid_n':hidden_layer_elements_list, 'n_hid':n_hidden_layers_list}

# Llamadas de ejemplo
# TODO - Sustituir por las llamadas correctas
plot_heatmap_with_values(resultados_validacion, grids, 'n_hid', 'hid_n', 'reg', 'relu')
plot_heatmap_with_values(resultados_validacion, grids, 'n_hid', 'reg', 'hid_n', 'relu', log_y=True)